# Ejercicio Módulo 2
**Inteligencia Artificial - CEIA - FIUBA**

**INSERTE AQUÍ SU NOMBRE**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [12]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi

In [14]:
def search_algorithm(number_disks=5) -> (NodeHanoi, dict):

    list_disks = [i for i in range(5, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    ##### EDITAR ESTA ZONA
    import heapq

    def heuristica_propia(node):
        """
        Heurística propia para A* en la Torre de Hanoi.
        Penaliza cada disco que NO está en el rod destino (rods[2])
        con su tamaño como peso. Los discos más grandes bloquean a
        los menores, por lo que su penalización es proporcionalmente mayor.

            h(n) = Σ tamaño(disco_i) para todo disco_i fuera del rod destino

        Admisible: nunca sobreestima, ya que mover un disco cuesta al menos 1
        y la suma de tamaños está siempre por debajo del costo real.
        Consistente: h(n) <= 1 + h(n') para cualquier sucesor n'.
        """
        discos_en_destino = set(goal_state.rods[2])
        h = 0
        for rod in node.state.rods:
            for disco in rod:
                if disco not in discos_en_destino:
                    h += disco
        return float(h)
    
    # Inicializamos las salidas, pero reemplazar con lo que se quiera usar.
    initial_node = NodeHanoi(initial_state)
    counter = 0
    frontier = []
    heapq.heappush(frontier, (
        initial_node.path_cost + heuristica_propia(initial_node),
        counter,
        initial_node
    ))

    explored = set()
    nodes_explored = 0
    max_depth = 0
    solution = None

    while frontier:
        _, _, node = heapq.heappop(frontier)

        if node.depth > max_depth:
            max_depth = node.depth

        if problem.goal_test(node.state):
            solution = node
            break

        state_key = str(node.state)
        if state_key in explored:
            continue
        explored.add(state_key)
        nodes_explored += 1

        for child in node.expand(problem):
            if str(child.state) not in explored:
                f = child.path_cost + heuristica_propia(child)
                counter += 1
                heapq.heappush(frontier, (f, counter, child))

    solution_found = solution is not None
    metrics = {
        "solution_found": solution_found,
        "nodes_explored": nodes_explored,
        "states_visited": len(explored),
        "nodes_in_frontier": len(frontier),
        "max_depth": max_depth,
        "cost_total": float(solution.path_cost) if solution_found else None,
    }

    if not solution_found:
        solution = None
        
    # TODO: Completar con el algoritmo de búsqueda que desees implementar
    #####

    return solution, metrics

Se prueba la función:

In [15]:
solution, metrics = search_algorithm(number_disks=5)

Veamos las métricas:

In [16]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 232
states_visited: 232
nodes_in_frontier: 33
max_depth: 31
cost_total: 31.0


Veamos el camino de estados desde el principio a la solución:

In [17]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [18]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
